[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/tennis_racket/tennis_racket.ipynb)

# The Tennis Racket Theorem

This example spins a box about each of its principal axes. Thrown spinning about its longest or its shortest axis, it keeps spinning; about the axis in between, the smallest wobble grows, and the box flips over and back, again and again, though nothing acts on it. This is the tennis racket theorem, and on the space station it is the Dzhanibekov effect.

A body's rate of turning is a bivector, the plane it turns in, and its momentum an antibivector. Inertia is the map between them, built from the body's points as `points & points.commutator(Rate)`, summed: the point joined with the velocity a rate gives it. Without forces the momentum is fixed in the world, `motor >> inertia(rate)`, while in the body the rate changes by the momentum's commutator with it, read back through the inverse inertia. Nothing here names the dimension: in four dimensions a box has six principal planes, and the same lines spin it in each.

In the matrix notation of rigid-body dynamics the rate reads as the angular velocity vector $\boldsymbol\omega$, the inertia as the tensor $I$, and torque-free motion as Euler's equations $I\dot{\boldsymbol\omega} = (I\boldsymbol\omega)\times\boldsymbol\omega$, which hold in three dimensions only.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra
from examples import instantiate
from examples.animation import save_figure
from examples.mechanics.tennis_racket import render, scenarios

np.set_printoptions(precision=4, suppress=True)

## 1. A box spun in each principal plane

The box is its corners, points of unit mass at distinct distances along each axis, so that it has distinct principal moments. Its inertia sends a rate to a momentum; the rate's kinetic energy is half the momentum paired with the rate. One copy of the box is spun in each principal plane, with a disturbance of one part in a hundred thousand. In such a plane the momentum lies along the rate, their commutator vanishes, and the rate does not change, but only the planes of the largest and smallest moment hold on to that.

In [ ]:
# The dimension of space, the time step and the number of steps, and the seed of the small disturbances.
dimension = 3
step = 0.25
n_steps = 800
seed = 42

In [ ]:
core = instantiate("examples.mechanics.tennis_racket.core", Algebra.from_pqr(dimension, 0, 0))
body = core.racket(seed)                                                       # [bodies] Body
momentum = body.inertia(body.rate)                                             # [bodies] Forque
energy = (momentum & body.rate) / 2                                            # [bodies] Scalar
# Euler's equation in the body: the momentum's commutator with the rate, back through the inverse inertia.
rate_change = body.inverse_inertia(-momentum.commutator(body.rate))            # [bodies] Rate

In [ ]:
print("kinetic energy of each body:", energy.to_array())
print("largest rate of change of the rate:", np.abs(rate_change.kernel).max())

## 2. The intermediate plane tumbles

Each body moves without forces: its motor steps by the exponential of its rate, and its rate by Euler's equation. In the plot every panel is one body, its rate's parts in the three planes over time. The bodies spun in the planes of the largest and smallest moment keep spinning as they started. The one spun in the intermediate plane holds for a while, then turns over completely, and back again, at regular intervals: the disturbance grows exponentially until the turn, which is a passage along a separatrix of the motion. Throughout, every body keeps its energy and its momentum in the world.

In [ ]:
motors, rates = core.simulate(body, core.lie.explicit_rkmk4, step, n_steps)       # [steps + 1, bodies] Motor, Rate
energies = core.lie.kinetic_energy(rates, body.inertia)                         # [steps + 1, bodies] Scalar
# The momentum in the world, carried out of the body by its motor.
world_momenta = motors >> body.inertia(rates)                                   # [steps + 1, bodies] Forque
# Whether each body's rate ever points against its start.
reversed_rates = (rates.dot(body.rate) < 0).any(axis=0)                         # [bodies]
save_figure(render.draw_trajectories(rates, energies), "tennis_racket_3d");

In [ ]:
print("rate reversed, per body:", reversed_rates)
print("largest change of the world momentum:", np.abs((world_momenta - world_momenta[0]).kernel).max())

## 3. In four dimensions

A box in four dimensions has four distinct moments and six principal planes, and the same lines spin it in each. Now more than one plane lies between others: the planes of the largest and smallest moments are stable again, and among the rest some tumble regularly, some tumble irregularly, and some hold.

In [ ]:
higher_dimension = 4

In [ ]:
higher_rates, higher_energies = scenarios.spinning_racket(higher_dimension, step, n_steps * step, seed)   # [steps + 1, bodies] Rate, Scalar
save_figure(render.draw_trajectories(higher_rates, higher_energies), "tennis_racket");

## 4. Keeping the momentum in the world

How well an integrator keeps the momentum fixed in the world depends on how it steps the motor. Stepping the rate by Runge–Kutta and then the motor by the exponential of the stepped rate, as Verlet and plain RK4 do here, lets the world momentum drift at first order. The Munthe-Kaas method steps a bivector in the Lie algebra instead, and the motor by its exponential: the bivector's rate is the body's rate corrected for the curvature of the exponential, a polynomial in the commutator with an open bivector, `h.commutator(Bivector)`, the same in every dimension. It keeps the world momentum to fourth order, in three, four and five dimensions alike.

In the notation of Lie-group integration that correction reads as the inverse of the derivative of the exponential map, $\operatorname{dexp}^{-1}_h = 1 - \tfrac12\operatorname{ad}_h + \tfrac1{12}\operatorname{ad}_h^2$, truncated where the fourth-order method needs it.

In [ ]:
compared_dimensions = (3, 4, 5)
comparison_runtime = 100.0

In [ ]:
curves = list(scenarios.integrator_comparison(compared_dimensions, step, comparison_runtime, seed))   # (dimension, integrator, drift) each
save_figure(render.draw_integrator_comparison(curves, step), "tennis_racket_integrators");

In [ ]:
for dimension_compared, name, drift in curves:
    print(f"{dimension_compared} dimensions, {name:6s}: worst world momentum drift {drift.to_array().max():.2e}")

In [ ]:
# checks
# The box spun in a principal plane does not change its rate; of the three spun in three dimensions only
# the intermediate one reverses; every body keeps its energy to the accuracy of the steps, and the
# Munthe-Kaas steps keep the world momentum far better than the others.
np.testing.assert_allclose(rate_change.kernel, 0.0, atol=1e-3)
assert reversed_rates.sum() == 1
np.testing.assert_allclose((energies / energies[0]).to_array(), 1.0, atol=1e-4)
worst = {(dimension_compared, name): drift.to_array().max() for dimension_compared, name, drift in curves}
for dimension_compared in compared_dimensions:
    assert worst[dimension_compared, "rkmk4"] < 0.1 * worst[dimension_compared, "verlet"]